## FormalVizWidget

A generic `anywidget` for animating an SVG whose visual state is driven by
a formal model. This notebook is a reusable framework for implementing animation for the spectabular specifcation libary. A
model specific example notebook is given in `elevator/Elevator.ipynb`.

To use, import the notebook with the `%run` shell magic

```
%run "../FormalVizWidget.ipynb"
```

and then supply:

- an SVG with element `id`s to animate,
- a `state` dict shape,
- a JS `animation_handler` function mapping `(prev_state, next_state)` to
  DOM/CSS updates,
- a Spectabular relation spec, as a class variable on a `FormalVizWidget`
  subclass.

In [1]:
import asyncio
import json
import time as _time

import anywidget
import traitlets


### The widget

`FormalVizWidget` renders `_svg_content` into the DOM once, then re-runs a
JS `animation_handler` every time the traitlets `state` dict changes.
Python sets `state` (with `transition`/`snap` below) and the JS side
interpolates between the old and new values using the `tween`/`snap`/
`setAttr` helpers passed into the handler.

`state` is just what's currently on screen. Each widget instance also
keeps its own local, Python-only history of how it got there
(`_history`/`_events`/`_cursor`, never synced to JS) so that:

- `w.snapshot()` can fork off a brand new, independent widget (its own
  model, its own DOM) from wherever `w` currently is, so two widgets
  displayed in different cells stop being two *views of the same model*
  (the usual ipywidgets/anywidget gotcha, where any state change updates
  every view) and become genuinely separate.
- `w.replay()`, `w.export_trace()`, `w.goto_transition(i)`, `w.prev()`,
  `w.next()` can navigate or dump that history without re-solving the
  spec, and without disturbing that history's ability to differ between
  `w` and any widget forked from it.


In [2]:
ESM = r"""
export default {
  render({ model, el }) {
    const container = document.createElement("div");
    el.appendChild(container);

    let svg = null;
    let handler = null;
    let prevState = null;

    // tween helper. via css.
    // useage: tween(element, { fill:"#ffaa00", fillOpacity:"0.9" }, 300)
    function tween(el, styles, durMs) {
      if (!el) return;
      const keys = Object.keys(styles);
      // dont pass in camelCase !!!!
      el.style.transition = keys.map(k => k + " " + durMs + "ms ease-in-out").join(", ");
      // apply new values on next frame so the transition triggers
      requestAnimationFrame(() => {
        for (const [k, v] of Object.entries(styles)) {
          el.style[k] = v;
        }
      });
    }
    
    // snap helper instant, no transition
    function snap(el, styles) {
      if (!el) return;
      el.style.transition = "none";
      for (const [k, v] of Object.entries(styles)) {
        el.style[k] = v;
      }
    }

    // SVG attr helper. to directly set svg attr instead of css
    function setAttr(el, attr, val) {
      if (!el) return;
      el.setAttribute(attr, val);
    }

    // build animation handler from user code
    function buildHandler() {
      const code = model.get("_animation_handler");
      if (!code) { handler = null; return; }
      try {
        // handler signature: (prev, next, svg, durationMs, tween, snap, setAttr)
        handler = new Function("prev", "next", "svg", "duration", "tween", "snap", "setAttr", code);
      } catch (e) {
        console.error("animation_handler compile error:", e);
        handler = null;
      }
    }

    // render SVG
    function renderSVG() {
      container.innerHTML = model.get("_svg_content");
      svg = container.querySelector("svg");
      if (!svg) return;
      // sometimes we want to change viewbox. look at elevator example
      const vb = model.get("_viewbox");
      if (vb) svg.setAttribute("viewBox", vb);
      const state = model.get("state");
      // apply init state
      if (handler && svg) {
        handler({}, state, svg, 0, snap, snap, setAttr);
      }
      prevState = { ...state }; //copy 
    }

    // when state changes, animate from prev -> next
    function onStateChange() {
      const nextState = model.get("state");
      const dur = model.get("_transition_duration");
      if (handler && svg && prevState) {
        handler(prevState, nextState, svg, dur, tween, snap, setAttr);
      }
      prevState = { ...nextState };
    }

    model.on("change:_svg_content", renderSVG);
    model.on("change:_animation_handler", () => { buildHandler(); renderSVG(); });
    model.on("change:state", onStateChange);

    buildHandler();
    renderSVG();
  }
};
"""

In [3]:
class FormalVizWidget(anywidget.AnyWidget):
    _esm = ESM

    _svg_content = traitlets.Unicode("").tag(sync=True)
    _viewbox = traitlets.Unicode("").tag(sync=True)
    _max_width = traitlets.Unicode("800px").tag(sync=True)
    _animation_handler = traitlets.Unicode("").tag(sync=True)
    _transition_duration = traitlets.Float(300).tag(sync=True)  # ms

    # the displayed state, mirrored to JS. always equal to
    # self._history[self._cursor]
    state = traitlets.Dict({}).tag(sync=True)

    # set by a subclass to a Spectabular relation spec: a `VectorTable`, or
    # a `Table` selecting among named relations, or
    # several combined with operators.
    # basically what people would expect
    spec = None

    def __init__(self, svg_content="", variables=None,
                 animation_handler="", viewbox=None, max_width="800px", **kw):
        initial = dict(variables or {})
        super().__init__(
            _svg_content=svg_content,
            state=dict(initial),
            _animation_handler=animation_handler,
            _viewbox=viewbox or "",
            _max_width=max_width,
            **kw, # allow user to edit anywidget settings
        )
        self._initial = initial
        # transition history: _history[i] is the state after _events[i-1]
        # fired (_history[0] is always the initial state, so
        # len(_history) == len(_events) + 1). _cursor indexes into
        # _history for whichever point is currently on screen, normally
        # the most recent entry, but goto_transition/prev/next can move it
        # back without discarding anything, so scrubbing through history
        # doesn't require re-running the formal model.
        self._history = [dict(initial)]
        self._events = []
        self._cursor = 0

    # make this from svg
    @classmethod
    def from_svg(cls, path, variables, animation_handler, viewbox=None, **kw):
        with open(path) as f:
            svg = f.read()
        return cls(svg, variables, animation_handler, viewbox, **kw)

    @property
    def s(self):
        return _S(self) #delegate

    # sets what's on screen without using history. used both by the
    # transition/snap methods below (which do record) and by history
    # navigation (which deliberately doesn't; moving around in time you
    # already have isn't a new transition).

    def _display(self, state, duration):
        self._transition_duration = duration * 1000  # ms for JS
        self.state = dict(state)

    def _record(self, next_state, event, label):
        # if the cursor isn't at the head (the user rewound with prev()/
        # goto_transition() and is now doing something new)
        del self._history[self._cursor + 1:]
        del self._events[self._cursor:]
        self._events.append(event if event is not None else label)
        self._history.append(dict(next_state))
        self._cursor = len(self._history) - 1

    def reset(self):
        # back to the initial state, and forget the history that led
        # anywhere else
        self._history = [dict(self._initial)]
        self._events = []
        self._cursor = 0
        self._display(self._initial, 0)

    async def transition(self, next_state, duration=0.3, event=None, label=None):
        # await until transition
        self._record(next_state, event, label)
        self._display(next_state, duration)
        await asyncio.sleep(duration)

    def transition_sync(self, next_state, duration=0.3, event=None, label=None):
        # block until transition
        self._record(next_state, event, label)
        self._display(next_state, duration)
        _time.sleep(duration)

    def snap(self, next_state, event=None, label=None):
        # snap
        self._record(next_state, event, label)
        self._display(next_state, 0)

    # move the display around in already-recorded history. non-animated
    # by default since these are typically single scrub steps run from
    # their own cell; pass duration for a tween instead.

    def goto_transition(self, index, duration=0.0):
        index = max(0, min(index, len(self._history) - 1))
        self._cursor = index
        self._display(self._history[index], duration)

    def prev(self, duration=0.0):
        if self._cursor > 0:
            self.goto_transition(self._cursor - 1, duration)

    def next(self, duration=0.0):
        if self._cursor < len(self._history) - 1:
            self.goto_transition(self._cursor + 1, duration)

    @property
    def cursor(self):
        return self._cursor

    @property
    def can_prev(self):
        return self._cursor > 0

    @property
    def can_next(self):
        return self._cursor < len(self._history) - 1

    @property
    def history(self):
        # states visited, index 0 is always the initial state
        return [dict(s) for s in self._history]

    def __len__(self):
        # number of recorded transitions (== len(history) - 1)
        return len(self._events)

    async def replay(self, tick_dt=0.5, transition_dur=0.15, from_index=0, to_index=None):
        """
        Re-animate through already-recorded history, in order, 
        without recomputing states
        """
        if to_index is None:
            to_index = len(self._history) - 1
        self.goto_transition(from_index, duration=0)
        for i in range(from_index + 1, to_index + 1):
            self.goto_transition(i, duration=transition_dur)
            await asyncio.sleep(transition_dur)
            await asyncio.sleep(max(0.0, tick_dt - transition_dur))

    def export_trace(self, as_json=False):
        trace = [
            {
                "index": i,
                "event": self._events[i],
                "before": self._history[i],
                "after": self._history[i + 1],
            }
            for i in range(len(self._events))
        ]
        return json.dumps(trace, default=str) if as_json else trace

    def snapshot(self):
        """
        A new, fully independent widget (its own model and DOM) showing
        this widget's current displayed state. Carries along the history up 
        to the cursor.
        """
        cls = type(self)
        copy = cls(
            svg_content=self._svg_content,
            variables=self._history[self._cursor],
            animation_handler=self._animation_handler,
            viewbox=self._viewbox,
            max_width=self._max_width,
        )
        copy._history = [dict(s) for s in self._history[:self._cursor + 1]]
        copy._events = list(self._events[:self._cursor])
        copy._cursor = self._cursor
        return copy

    @classmethod
    def _flattened_spec(cls):
        """
        `cls.spec`, flattened and indexed by free variable name, cached on
        the concrete subclass.
        had to use cls.__dict__ instead of hasattr to prevent shadowing from parent
        """
        if "_flat_spec_cache" not in cls.__dict__:
            if cls.spec is None:
                raise NotImplementedError(f"{cls.__name__}.spec is not set")
            flat = flatten(cls.spec)
            free_by_name = {str(v): v for v in freevars(flat)}
            if isinstance(cls.spec, VectorTable):
                # The variables the pecs assigns values to, one per row of
                # .left. primed (a relation table's next state values) or
                # unprimed (a predicate table's values).
                output_names = {str(v) for v in cls.spec.left}
            else:
                # No `.left` to read (like a Table()).
                # a free variable ending in `ʹ` is a next-state output, everything
                # else a read-only input.
                output_names = {name for name in free_by_name if name.endswith("ʹ")}
            cls._flat_spec_cache = (flat, free_by_name, output_names)
        return cls._flat_spec_cache

    @classmethod
    def compute_next_state(cls, state, **event):
        """
        Solve `cls.spec` for the values of whatever it assigns to
        given `state` plus whatever else a particular row needs.

        Works uniformly for both roles a VectorTable can play:
        - a relation table, where `.left` holds primed (`xʹ`) next-state
          variables. This is the usual case,the result is a next state,
          keyed by the unprimed names.
        - a **predicate table**, where `.left` holds unprimed
          variables derived from the current state. Itreturns
          those derived values, keyed by their own names.

        Every other free variable in the spec is a read only input.
        We pin from `state` if present there, else from `event` if supplied.
        An unprimed variable that's the counterpart of a primed output (i.e.
        clearly meant to be persisted state) must be in `state`, or this
        raises `KeyError`. Any other input (an event selector, a
        predicate's own extra parameter, ...) is simply left unconstrained
        if missing. Harmless if the row that fires doesn't actually
        depend on it.

        A classmethod rather than an instance method (and `spec` a class
        variable rather than instance state) so it also works without a
        live widget.
        """
        flat, free_by_name, output_names = cls._flattened_spec()
        #print("flat:", flat)
        #print("free: ", free_by_name)
        #print("output_names:", output_names)
        input_names = set(free_by_name) - output_names
        required_state_names = {name.removesuffix("ʹ") for name in output_names if name.endswith("ʹ")}
        #print(required_state_names)

        solver = z3.Solver()
        solver.add(flat)
        for name in input_names:
            if name in state:
                # read any free vars from state
                value = state[name]
            elif name in event:
                # read any free vars from event
                value = event[name]
            elif name in required_state_names:
                raise KeyError(f"state is missing {name!r}, required by {cls.__name__}.spec")
            else:
                continue  # not relevant to whichever row fires; let Z3 pick freely
            var = free_by_name[name]
            solver.add(var == _state_value_to_z3(value, var.sort()))

        result = solver.check()
        if result != z3.sat:
            raise ValueError(
                f"{cls.__name__}.spec has no enabled row for {event!r} from state {state!r} ({result})"
            )
        # a table can have several satisfying assignments for its
        # output variables (nondeterminism). this returns whichever one
        # Z3 happens to find, which is fine for scripted/replayed
        # animation (the scenario already picked the transition/row via
        # `event`) but not for interactively exploring all enabled next
        # states, which would need re-solving under a blocking clause per
        # answer found.
        model = solver.model()
        return {
            name.removesuffix("ʹ"): _z3_to_state_value(model.eval(free_by_name[name], model_completion=True))
            for name in output_names
        }


class _S:
    # attribute access of a widget's state (w.s.blinkRight instead of w.state["blinkRight"]).

    def __init__(self, widget):
        object.__setattr__(self, "_widget", widget)

    def __getattr__(self, name):
        try:
            return self._widget.state[name]
        except KeyError:
            raise AttributeError(name) from None

    def __setattr__(self, name, value):
        # allows us to use the ** syntax. goes through snap() (not the
        # trait directly) so direct field edits still land in history.
        next_state = dict(self._widget.state)
        next_state[name] = value
        self._widget.snap(next_state, event={name: value})

    def __repr__(self):
        return repr(self._widget.state)


def _state_value_to_z3(value, sort):
    if isinstance(value, bool):
        return z3.BoolVal(value)
    if isinstance(value, int):
        return z3.IntVal(value)
    if isinstance(value, float):
        return z3.RealVal(value)
    if isinstance(value, str):
        for i in range(sort.num_constructors()):
            if sort.constructor(i).name() == value:
                return sort.constructor(i)()
        raise ValueError(f"{value!r} is not a value of enum sort {sort}")
    raise TypeError(f"don't know how to convert {value!r} (a {type(value).__name__}) to a Z3 term")


def _z3_to_state_value(value):
    kind = value.sort().kind()
    if kind == z3.Z3_BOOL_SORT:
        return z3.is_true(value)
    if kind == z3.Z3_INT_SORT:
        return value.as_long()
    if kind == z3.Z3_REAL_SORT:
        return value.as_fraction()
    return str(value)  # enum (or other datatype) constructor name


TODO:
- Expose multiple satisfying next states for a nondeterministic relation
  (re-solve per model found) to support
  interactive exploration rather than only scripted replay.
- Catch trying to use 2D Tables in the animation. Maybe we put this onus on the animator?

### Scenario runners
Helpers to run scenarios.

In [4]:
async def run_scenario(w, events, tick_dt=0.5, transition_dur=0.15):
    """
    Run a sequence of events, animating smoothly between each.

    w : FormalVizWidget
    events : list of dicts, one set of `compute_next_state` keyword arguments per event
    tick_dt : seconds to wait after each event
    transition_dur : animation duration per transition
    """
    for event in events:
        next_state = w.compute_next_state(w.state, **event)
        await w.transition(next_state, duration=transition_dur, event=event)
        await asyncio.sleep(max(0.0, tick_dt - transition_dur))


async def advance_ticks(w, n, event, tick_dt=0.5, transition_dur=0.15, stop_when=None):
    """
    Send the same event `n` times in a row animating between each.
    Stops early if `stop_when(state)` becomes true. Usefull for
    modeling time based applications

    Returns the number of events actually sent.
    """
    for i in range(n):
        if stop_when is not None and stop_when(w.state):
            return i
        next_state = w.compute_next_state(w.state, **event)
        await w.transition(next_state, duration=transition_dur, event=event)
        await asyncio.sleep(max(0.0, tick_dt - transition_dur))
    return n


### General helpers
useful for debugging or testing any `compute_next_state` function against any widget.

In [5]:
def diff_state(before, next):
    # return what varibles in state change from before to next
    keys = before.keys() | next.keys()
    return {k: (before.get(k), next.get(k)) for k in keys if before.get(k) != next.get(k)}


def trace_scenario(widget_cls, initial_state, events):
    # tracer with no animation
    state = dict(initial_state)
    trace = []
    for event in events:
        next_state = widget_cls.compute_next_state(state, **event)
        trace.append((event, dict(state), dict(next_state)))
        state = next_state
    return trace


async def reset_and_run(w, scenario_fn, *args, settle=0.3, **kwargs):
    """
    Reset a widget to its initial state, let the reset render settle, then
    run `scenario_fn(w, *args, **kwargs)` (e.g. a demo_* function or
    `run_scenario`). 
    """
    w.reset()
    await asyncio.sleep(settle)
    await scenario_fn(w, *args, **kwargs)

### Using this framework

This notebook defines `FormalVizWidget` (with its generic
`compute_next_state`), `run_scenario`, `advance_ticks`, `diff_state`,
`trace_scenario`, and `reset_and_run` in the calling kernel once `%run`
has executed it. A model notebook:

1. `%run`s this notebook, then `%run`s `spectabular.ipynb` for
   `Bool`/`Int`/`EnumType`/`VectorTable`/`flatten`/...
2. builds its spec and subclasses `FormalVizWidget` with `spec = ...` set
   as a class variable.

See `elevator/Elevator.ipynb` for a full worked example.

#### Changing state

- `w.transition(next_state, duration=0.3, event=None, label=None)`.
  async, animates and records a new step in `w`'s history.
- `w.transition_sync(...)`. same, but blocks instead of `await`ing.
- `w.snap(next_state, event=None, label=None)`. instant, still recorded.
- `w.s.name = value`. shorthand for a one-field `snap`; also recorded.
- `w.reset()`. back to the initial state, discarding all history.

`event`/`label` is whatever you want attached to that step for
`export_trace()` later, typically the same kwargs dict passed to
`compute_next_state`, which is what `run_scenario`/`advance_ticks` record
automatically.

#### Forking: `snapshot()`

```python
w1 = w.snapshot()   # independent widget, currently identical to w
w.transition(...)   # only w moves
w1.transition(...)  # only w1 moves, from where it was snapshotted
```

`snapshot()` returns a new widget of the same class, showing wherever
`w`'s cursor currently is (not necessarily its latest state, see
below), carrying along the history that led there. From that point on
the two widgets never affect each other.

#### Transition history

- `w.history` - list of every state visited, `history[0]` is the
  initial state.
- `w.cursor`, `len(w)` - current position, and number of recorded
  transitions.
- `w.export_trace(as_json=False)` - list (or JSON string) of
  `{index, event, before, after}` per transition.
- `w.goto_transition(i, duration=0.0)` - jump the display to
  `history[i]` without re-solving anything or changing what's recorded.
- `w.prev(duration=0.0)` / `w.next(duration=0.0)` - step the cursor by
  one; `w.can_prev` / `w.can_next` say whether there's anywhere to go.
- `await w.replay(tick_dt=0.5, transition_dur=0.15, from_index=0, to_index=None)`
  - re-animate through recorded history in order. Unlike re-running
  `run_scenario`, this never calls `compute_next_state` again, so it
  reproduces exactly what happened even if the spec is nondeterministic.

Transitioning after rewinding with `prev()`/`goto_transition()` truncates
whatever history was ahead of the cursor before recording new stuff. `w.snapshot()` can
preserve a branch point instead of overwriting it.
